In [1]:
import ants
import torch
import torchio as tio
import random
import numpy as np
import torchio.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
array = np.zeros((9, 9, 9))
array[3:6, 3:6, 3] = 1
array[0, 0, 0] = 1
array[5, 0, 5] = 1

array=torch.tensor(array).unsqueeze(0)
print(array.shape)
print(array)
# Set seeds for reproducibility
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
np.random.seed(0)
random.seed(0)

torch.Size([1, 9, 9, 9])
tensor([[[[1., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [

In [3]:
new_data=tio.LabelMap(tensor=array)

subject_dict = {
    'mask': new_data,
}

subject = tio.Subject(subject_dict)

dataset = tio.SubjectsDataset([subject])

label_sampler = tio.data.LabelSampler(patch_size=3,
                                    label_name='mask', 
                                    label_probabilities={0: 3, 1: 4})

patches_training_set=tio.Queue(
    dataset,
    max_length=50,
    samples_per_volume=10,
    sampler=label_sampler,
    num_workers=1,
    shuffle_subjects=True,
    shuffle_patches=True,
)

In [4]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Assuming patches_training_set is defined
training_loader = DataLoader(patches_training_set, batch_size=4, shuffle=True)

for batch in training_loader:
    masks = batch['mask']['data']  # Assuming the key for the images is 'mask'
    masks_np = masks.numpy()  # Convert to NumPy array for easier slicing and visualization
    
    # Determine the number of patches in the batch
    num_patches = masks_np.shape[0]
    
    def update_plot(slice_index):
        fig, axs = plt.subplots(1, num_patches, figsize=(15, num_patches * 3))
        for i in range(num_patches):
            slice_2d = masks_np[i, 0, slice_index, :, :]  # Use the slider's value to select the slice
            axs[i].imshow(slice_2d, cmap='gray')
            axs[i].axis('off')
        plt.show()

    # Assuming the patches are 3D, create a slider to select the slice
    slice_slider = widgets.IntSlider(min=0, max=masks_np.shape[2]-1, step=1, value=masks_np.shape[2] // 2, description='Slice')
    
    # Display the widget and link it to the update function
    interactive_plot = widgets.interactive(update_plot, slice_index=slice_slider)
    display(interactive_plot)

    # Break after the first batch to avoid creating widgets for all batches
    #break

interactive(children=(IntSlider(value=1, description='Slice', max=2), Output()), _dom_classes=('widget-interac…

interactive(children=(IntSlider(value=1, description='Slice', max=2), Output()), _dom_classes=('widget-interac…

interactive(children=(IntSlider(value=1, description='Slice', max=2), Output()), _dom_classes=('widget-interac…

New LABELSAMPLER


In [6]:
label_sampler = tio.data.sampler.LabelSampler_Pad4LabelPatches(
    patch_size=3,
    label_name='mask',
    label_probabilities={0: 3, 1: 4},

)

patches_training_set=tio.Queue(
    dataset,
    max_length=50,
    samples_per_volume=10,
    sampler=label_sampler,
    num_workers=1,
    shuffle_subjects=True,
    shuffle_patches=True,
)



In [7]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Assuming patches_training_set is defined
training_loader = DataLoader(patches_training_set, batch_size=4, shuffle=True)

for batch in training_loader:
    masks = batch['mask']['data']  # Assuming the key for the images is 'mask'
    masks_np = masks.numpy()  # Convert to NumPy array for easier slicing and visualization
    
    # Determine the number of patches in the batch
    num_patches = masks_np.shape[0]
    
    def update_plot(slice_index):
        fig, axs = plt.subplots(1, num_patches, figsize=(15, num_patches * 3))
        for i in range(num_patches):
            slice_2d = masks_np[i, 0, slice_index, :, :]  # Use the slider's value to select the slice
            axs[i].imshow(slice_2d, cmap='gray')
            axs[i].axis('off')
        plt.show()

    # Assuming the patches are 3D, create a slider to select the slice
    slice_slider = widgets.IntSlider(min=0, max=masks_np.shape[2]-1, step=1, value=masks_np.shape[2] // 2, description='Slice')
    
    # Display the widget and link it to the update function
    interactive_plot = widgets.interactive(update_plot, slice_index=slice_slider)
    display(interactive_plot)

    # Break after the first batch to avoid creating widgets for all batches
    #break

ModuleNotFoundError: No module named 'torchio.transforms.preprocessing.spatial.compose'